### RAG DATA INGESTION PIPELINE


In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

from pathlib import Path

C:\Users\Admin\AppData\Local\Temp\ipykernel_5196\2539874464.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [5]:
def load_pdf(dir):
    directory = Path(dir)
    pdf_docs_list = []

    pdf_files = list(directory.glob("**/*.pdf"))

    for each_file in pdf_files:
        try:
            loader = PyPDFLoader(each_file)
            parsed = loader.load()
            pdf_docs_list.extend(parsed)
        except Exception as e:
            print(f"Error: {e}")

    return pdf_docs_list


pdf_documents = load_pdf("../data/pdf_files")
print(len(pdf_documents))

4


### TEXT SPLITTING AND CHUNKING


In [6]:
### TEXT SPLITTING AND CHUNKING
from langchain_text_splitters import RecursiveCharacterTextSplitter


def custom_document_splitter(docs):
    """Split Docs into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    splitted_docs = text_splitter.split_documents(docs)
    print(f"Splitted { len(docs) } docs into {len(splitted_docs)} chunks")

    if splitted_docs:
        print(f"\n Example Chunk ")
        print(f"Content: {splitted_docs[0].page_content[:200]}..")
        print(f"Metadata: {splitted_docs[0].metadata}")
    return splitted_docs


chunks = custom_document_splitter(pdf_documents)
print(chunks)

Splitted 4 docs into 92 chunks

 Example Chunk 
Content: Deep Learning and Neural Networks..
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-05T10:13:52+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-05T10:13:52+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\deep_learning_neural_networks.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}
[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-05T10:13:52+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-05T10:13:52+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\deep_learning_neural_networks.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Deep Learning and Neural Networks'), Document(metadata={'producer': 'Re

#### EMBEDDING


In [7]:
# EMBEDDING
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Admin\Desktop\Python\rag\rag-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2070.37it/s]


### VECTOR DATABASE STORING


In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_database()

    def _initialize_database(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents_into_vectordb(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 552


In [10]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents_into_vectordb(chunks, embeddings)

print("Data Ingestion is finished")

Generating embeddings for 92 texts...


Batches: 100%|██████████| 3/3 [00:01<00:00,  1.95it/s]


Generated embeddings with shape: (92, 384)
Adding 92 documents to vector store...
Successfully added 92 documents to vector store
Total documents in collection: 644
Data Ingestion is finished


### RAG Retrieval


In [17]:
class RAG_retriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")  # Asking user query
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding // query will be embedded
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # and Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAG_retriever(vectorstore, embedding_manager)
rag_retriever.retrieve("Explain Machine Learning ")

Retrieving documents for query: 'Explain Machine Learning '
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7828d7db_26',
  'content': 'Machine Learning is a branch of artificial intelligence that allows computers to learn patterns',
  'metadata': {'source': '..\\data\\pdf_files\\machine_learning_ai.pdf',
   'creator': '(unspecified)',
   'title': '(anonymous)',
   'trapped': '/False',
   'page_label': '1',
   'keywords': '',
   'author': '(anonymous)',
   'doc_index': 26,
   'producer': 'ReportLab PDF Library - (opensource)',
   'creationdate': '2026-10-05T10:04:42+00:00',
   'moddate': '2026-10-05T10:04:42+00:00',
   'content_length': 95,
   'page': 0,
   'subject': '(unspecified)',
   'total_pages': 1},
  'similarity_score': 0.4528822898864746,
  'distance': 0.5471177101135254,
  'rank': 1},
 {'id': 'doc_7bf4d358_26',
  'content': 'Machine Learning is a branch of artificial intelligence that allows computers to learn patterns',
  'metadata': {'subject': '(unspecified)',
   'producer': 'ReportLab PDF Library - (opensource)',
   'moddate': '2026-10-05T10:04:42+00:00',
   'conte